# Problem Statement

## Business Context

As organizations grow and scale, they are often inundated with large volumes of data, reports, and documents that contain critical information for decision-making. In real-world business settings, such as venture capital firms like Andreesen Horowitz, business analysts are required to sift through large datasets, research papers, or reports to extract relevant information that impacts strategic decisions.

For instance, consider that you've just joined Andreesen Horowitz, a renowned venture capital firm, and you are tasked with analyzing a dense report like the Harvard Business Review's **"How Apple is Organized for Innovation."** Going through the report manually can be extremely time-consuming as the size and complexity of these report increases. However, by using **Semantic Search** and **Retrieval-Augmented Generation (RAG)** models, you can significantly streamline this process.

Imagine having the capability to directly ask questions like, “How does Apple structure its teams for innovation?” and get immediate, relevant answers drawn from the report. This ability to extract and organize specific insights quickly and accurately enables you to focus on higher-level analysis and decision-making, rather than being bogged down by information retrieval.

## Objective

The goal is to develop a RAG application that helps business analysts efficiently extract key insights from extensive reports, such as “How Apple is Organized for Innovation.”

Specifically, the system aims to:

- Answer user queries by retrieving relevant content directly from lengthy documents.

- Support natural-language interaction without requiring a full manual read-through.

- Act as an intelligent assistant that streamlines the report analysis process.

Through this solution, analysts can save time, improve productivity, and make faster, more informed strategic decisions

## Data Description

**How Apple is Organized for Innovation** - An article of 11 pages in pdf format

## Installing and Importing Necessary Libraries and Dependencies

In [34]:
# Install required libraries
!pip install -q langchain_community \
                langchain \
                langchain-text-splitters \
                chromadb \
                pymupdf \
                tiktoken \
                datasets \
                evaluate \
                langchain_openai \
                openai \
                rank_bm25

In [35]:
import os
import json

# Using OpenAI
from openai import OpenAI

# Working with PDFs
from langchain_community.document_loaders import PyMuPDFLoader

# Text Tokenizer
import tiktoken

# Chunking
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Embedding
from langchain_openai import OpenAIEmbeddings

# Vector datastore
from langchain_community.vectorstores import Chroma

# Hybrid retrieval
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever

In [36]:
# Load JSON config file and extract API credentials
file_name = 'config.json'
with open(file_name, 'r') as file:
    config = json.load(file)
    API_KEY = config.get("OPENAI_API_KEY")
    OPENAI_API_BASE = config.get("OPENAI_API_BASE")

# Store API credentials in os environment variables
os.environ['OPENAI_API_KEY'] = API_KEY
os.environ["OPENAI_BASE_URL"] = OPENAI_API_BASE

# Initialization of OpenAI client
client = OpenAI()

### Questions

In [37]:
question_1 = "Who are the authors of this article and who published this article?"
question_2 = "List down the three leadership characteristics in bulleted points and explain each one of the characteristics under two lines."
question_3 = "Can you explain specific examples from the article where Apple's approach to leadership has led to successful innovations?"

## Question Answering using LLM

In [38]:
# LLM raw response function (without prompt engineering)
def llm_response(user_prompt, max_tokens=1000, temperature=0.75, top_p=0.95):

    # Create a chat completion using the OpenAI client
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "user", "content": user_prompt}
        ],
        max_tokens=max_tokens,                                                  # Max number of tokens to generate in the response
        temperature=temperature,                                                # Controls randomness in output
        top_p=top_p                                                             # Controls diversity via nucleus sampling
    )
    return response.choices[0].message.content

### Question 1: Who are the authors of this article and who published this article?

In [39]:
llm_response_1 = llm_response(question_1)
print(llm_response_1)

I apologize, but I don't have access to external articles or databases to check specific articles, their authors, or publication details. If you provide the title or more context about the article, I may be able to help you with general information or guide you on how to find it.


### Question 2: List down the three leadership characteristics in bulleted points and explain each one of the characteristics under two lines.

In [40]:
llm_response_2 = llm_response(question_2)
print(llm_response_2)

- **Vision**: Effective leaders possess a clear vision of the future and can articulate it compellingly to inspire and motivate their team. This forward-thinking perspective helps guide decision-making and fosters a sense of purpose among team members.

- **Empathy**: Great leaders demonstrate empathy by understanding and valuing the feelings and perspectives of others. This emotional intelligence builds trust and rapport, enabling stronger relationships and a more cohesive team environment.

- **Decisiveness**: Strong leaders are able to make informed decisions promptly, even in uncertain situations. This decisiveness instills confidence in their team, ensuring that everyone is aligned and ready to move forward with clarity and purpose.


### Question 3: Can you explain specific examples from the article where Apple's approach to leadership has led to successful innovations?

In [41]:
llm_response_3 = llm_response(question_3)
print(llm_response_3)

I don’t have access to specific articles, but I can provide a general overview of how Apple's approach to leadership has contributed to its successful innovations based on widely recognized practices and examples.

1. **Visionary Leadership**: Under the leadership of Steve Jobs, Apple focused on creating products that combined technology with aesthetics. His vision for products like the iPod, iPhone, and iPad pushed the boundaries of design and functionality, resulting in transformative innovations in the tech industry.

2. **Cross-Functional Collaboration**: Apple fosters a culture of collaboration across different departments. For instance, the development of the Apple Watch involved input from design, engineering, and health teams to create a product that not only functions as a smartwatch but also integrates health monitoring features effectively.

3. **User-Centric Design**: Apple's emphasis on user experience has led to innovations like the intuitive interface of iOS. The seamles

## Question Answering using LLM with Prompt Engineering

In [42]:
# LLM System Prompt
llm_system_prompt = """
You are an AI assistant designed to support document analysis and to extract key insights.
Your task is to provide evidence-based, concise, and relevant information to user's question.

Here is an example of how to structure your response:

Answer:
[Key insight based on context]
"""

In [43]:
# LLM response function enhanced with prompt engineering
def eng_response(user_prompt, max_tokens=100, temperature=0.75, top_p=0.95):
  global llm_system_prompt

  # Create a chat completion using the OpenAI client
  response = client.chat.completions.create(
      model="gpt-4o-mini",
      messages=[
        {"role": "system", "content": llm_system_prompt},
        {"role": "user", "content": user_prompt}
      ],
      max_tokens=max_tokens,                                                    # Max number of tokens to generate in the response
      temperature=temperature,                                                  # Controls randomness in output (0 = deterministic)
      top_p=top_p                                                               # Controls diversity via nucleus sampling
  )
  return response.choices[0].message.content

### Question 1: Who are the authors of this article and who published this article?

In [44]:
eng_response_1 = eng_response(question_1)
print(eng_response_1)

Answer:
To provide the authors and publisher of the article, I would need specific details about the article in question, such as the title or subject matter. Please provide that information for accurate assistance.


### Question 2: List down the three leadership characteristics in bulleted points and explain each one of the characteristics under two lines.

In [45]:
eng_response_2 = eng_response(question_2)
print(eng_response_2)

Answer:
- **Visionary Thinking**: This characteristic involves the ability to create a clear, compelling vision for the future, motivating others to work towards shared goals and aspirations.

- **Emotional Intelligence**: Leaders with high emotional intelligence can understand and manage their own emotions and those of others, fostering strong relationships and effective communication within their teams.

- **Decisiveness**: This trait reflects the ability to make informed and timely decisions, often under pressure, which instills confidence in team members and


### Question 3: Can you explain specific examples from the article where Apple's approach to leadership has led to successful innovations?

In [46]:
eng_response_3 = eng_response(question_3)
print(eng_response_3)

Answer:
Apple's approach to leadership, particularly under Steve Jobs and later Tim Cook, has fostered successful innovations through a focus on design, user experience, and a strong organizational culture. 

1. **Design and User Experience**: Apple's emphasis on aesthetic design and intuitive user interfaces can be seen in products like the iPhone and MacBook. The leadership's insistence on high standards for product design has set Apple apart in a crowded market, leading to innovations that prioritize both functionality and style.

2


## Data Preparation for RAG

### Loading the Data

In [47]:
# Access, load, and extract content from the PDF
report_document = "HBR_How_Apple_Is_Organized_For_Innovation.pdf"
pdf_loader = PyMuPDFLoader(report_document)

### Data Overview

In [48]:
# Inspect loaded content
content = pdf_loader.load()
print(
    f"{len(content)} pages\n\n"
    f"First page: \n\n{content[0].page_content}"
)

11 pages

First page: 

REPRINT R2006F
PUBLISHED IN HBR
NOVEMBER–DECEMBER 2020
ARTICLE
ORGANIZATIONAL CULTURE
How Apple Is 
Organized  
for Innovation
It’s about experts leading experts. 
by Joel M. Podolny and Morten T. Hansen
This article is made available to you with compliments of Apple Inc for your personal use. Further posting, copying or distribution is not permitted.


### Data Chunking

In [49]:
# Initialize text splitter, chunk size and chunk overlap (for context continuity)
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name='cl100k_base',
    chunk_size=256,
    chunk_overlap=20
)

# Divide loaded content into chunks
document_chunks = pdf_loader.load_and_split(text_splitter)

### Embedding

In [50]:
# Initialize the OpenAI Embeddings model with API credentials
embedding_model = OpenAIEmbeddings(
    openai_api_key=API_KEY,                                                     # OpenAI API authentication key
    openai_api_base=OPENAI_API_BASE                                             # OpenAI API base URL endpoint
)

### Vector Database

In [51]:
# Build vector store
vectorstore = Chroma.from_documents(
    document_chunks,
    embedding_model
)

### Retriever

In [52]:
# Retriever wrapper for Vector DB to fetch 3 most relevant documents for a given query using similarity search
retriever = vectorstore.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 3}
)

### Response Function

In [53]:
# RAG System Prompt
rag_system_prompt = """
You are an AI assistant designed to support document analysis and to extract key insights.
Your task is to provide evidence-based, concise, and relevant information to consultant's question based on the context provided.

User input will include the necessary context for you to answer their questions. This context will begin with the token: ###Context.
The context contains references to specific portions of trusted literature and research articles relevant to the query, along with their source details.

When crafting your response:
1. Use only the provided context to answer the question.
2. If the answer is found in the context, respond with concise and actionable insights.
3. Include the source reference with the page number, journal name, or publication, as provided in the context.
4. If the question is unrelated to the context or the context is empty, clearly respond with: "Sorry, this is out of my knowledge base."

Please adhere to the following response guidelines:
- Provide clear, direct answers using only the given context.
- Do not include any additional information outside of the context.
- Avoid rephrasing or summarizing the context unless explicitly relevant to the question.
- If no relevant answer exists in the context, respond with: "Sorry, this is out of my knowledge base."
- If the context is not provided, your response should also be: "Sorry, this is out of my knowledge base."

Here is an example of how to structure your response:

Answer:
[Key insight based on context]

Source:
[Source details with page or section]
"""

In [54]:
# RAG response function enhanced with prompt engineering
def rag_response(user_prompt, k=3, max_tokens=1000, temperature=0.75, top_p=0.95):
    global rag_system_prompt

    # Retrieve relevant document chunks using vector store with configurable k
    relevant_document_chunks = vectorstore.similarity_search(user_prompt, k=k)
    context_list = [d.page_content for d in relevant_document_chunks]

    # Always include first chunk (title page with authors, publisher) - semantic search
    # often misses metadata-heavy content that doesn't embed well
    first_chunk_content = document_chunks[0].page_content
    if first_chunk_content not in context_list:
        context_list = [first_chunk_content] + context_list

    context_for_query = ". ".join(context_list)

    # Enhance user prompt
    rag_user_prompt = """
    ###Context
    Here are some excerpts from literature and their sources that are relevant to the question mentioned below:
    {context}

    ###Question
    {question}
    """
    rag_user_prompt = rag_user_prompt.replace('{context}', context_for_query)
    rag_user_prompt = rag_user_prompt.replace('{question}', user_prompt)

    # Create a chat completion using the OpenAI client
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": rag_system_prompt},
            {"role": "user", "content": rag_user_prompt}
        ],
        max_tokens=max_tokens,                                                  # Max number of tokens to generate in the response
        temperature=temperature,                                                # Controls randomness in output (0 = deterministic)
        top_p=top_p                                                             # Controls diversity via nucleus sampling
    )

    return response.choices[0].message.content.strip()

## Question Answering using RAG

### Question 1: Who are the authors of this article and who published this article?

In [55]:
rag_response_1 = rag_response(question_1, k=6)
print(rag_response_1)

Answer:
The authors of the article are Joel M. Podolny and Morten T. Hansen, and it was published by Harvard Business Review.

Source:
Harvard Business Review, November–December 2020.


### Question 2: List down the three leadership characteristics in bulleted points and explain each one of the characteristics under two lines.

In [56]:
rag_response_2 = rag_response(question_2)
print(rag_response_2)

Answer:
- **Deep expertise**: Leaders must have a thorough understanding of their individual functions, enabling them to engage meaningfully in the work and make informed decisions.
- **Immersion in details**: Leaders are expected to be deeply involved in the specifics of their functions, which helps them to identify issues and opportunities effectively.
- **Collaborative debate**: Leaders should be willing to engage in open discussions and debates with others, advocating for their views while remaining open to changing their stance based on new evidence.

Source:
Podolny, J. M., & Hansen, M. T. (2020). How Apple Is Organized for Innovation. Harvard Business Review.


### Question 3: Can you explain specific examples from the article where Apple's approach to leadership has led to successful innovations?

In [57]:
rag_response_3 = rag_response(question_3)
print(rag_response_3)

Answer:
The article highlights that Apple's organizational model, which is centered around functions rather than traditional business units, allows leaders to be cross-functionally collaborative and deeply knowledgeable. This structure aligns expertise with decision rights, facilitating informed decision-making that spurs innovation. A specific example includes Steve Jobs' restructuring of the company upon his return in 1997, where he eliminated the general manager positions of the business units. This move aimed to reduce internal competition and conflicts over resource allocations, fostering a more cohesive environment for innovation.

Source:
Harvard Business Review, November–December 2020, "How Apple Is Organized for Innovation" by Joel M. Podolny and Morten T. Hansen.


## Actionable Insights and Business Recommendations

In [58]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
evaluate_llm = ChatOpenAI(model_name="gpt-4o")

In [59]:
def response_evaluation(content, question, response):

    # Evaluate response accuracy
    evaluation_prompt = f"""
    Evaluate the assistant's response to a user's query using the provided context.

    Context: {content}
    Query: {question}
    Response: {response}

    Instructions:
    1. **Groundedness (0.0 to 1.0)**: Score based on how well the response is factually supported by the context.
                                    - Score closer to 1 if all facts are accurate and derived from the context.
                                    - Score closer to 0 if there is hallucination, guesswork, or any fabricated information.

    2. **Precision (0.0 to 1.0)**: Score based on how directly and accurately the assistant addresses the query.
                                    - Score closer to 1 if the response is concise, focused, and answers the exact user query.
                                    - Score closer to 0 if it includes irrelevant details or misses the main point.

    Output format:
      groundedness: float between 0 and 1 ,
      precision: float between 0 and 1

    """

    response_evaluation = evaluate_llm.invoke([HumanMessage(content=evaluation_prompt)]).content.strip()

    return response_evaluation

In [60]:
print(
    f"Question 1: {question_1}\n"
    f"llm_response: {response_evaluation(content, question_1, llm_response_1)}\n"
    f"eng_response: {response_evaluation(content, question_1, eng_response_1)}\n"
    f"rag_response: {response_evaluation(content, question_1, rag_response_1)}"
)

Question 1: Who are the authors of this article and who published this article?
llm_response: groundedness: 0.0, precision: 0.0
eng_response: groundedness: 0.0, precision: 0.0
rag_response: groundedness: 1.0,  
precision: 1.0


In [61]:
print(
    f"Question 2: {question_2}\n"
    f"llm_response: {response_evaluation(content, question_2, llm_response_2)}\n"
    f"eng_response: {response_evaluation(content, question_2, eng_response_2)}\n"
    f"rag_response: {response_evaluation(content, question_2, rag_response_2)}"
)

Question 2: List down the three leadership characteristics in bulleted points and explain each one of the characteristics under two lines.
llm_response: groundedness: 0.0, precision: 0.0
eng_response: groundedness: 0.0, precision: 0.0
rag_response: groundedness: 1.0, precision: 1.0


In [62]:
print(
    f"Question 3: {question_3}\n"
    f"llm_response: {response_evaluation(content, question_3, llm_response_3)}\n"
    f"eng_response: {response_evaluation(content, question_3, eng_response_3)}\n"
    f"rag_response: {response_evaluation(content, question_3, rag_response_3)}"
)

Question 3: Can you explain specific examples from the article where Apple's approach to leadership has led to successful innovations?
llm_response: groundedness: 0.2, precision: 0.3
eng_response: groundedness: 0.2, precision: 0.3
rag_response: groundedness: 0.7, precision: 0.6
